In [ ]:
# =============================================================================
# Experiment 7: Solving CAPTCHA Challenges using Machine Learning
# Objective: Build a machine learning model to solve CAPTCHA challenges. [CO3]
# =============================================================================

# 1. Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import random
import string
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =============================================================================
# 2. Generate Synthetic 4-Character CAPTCHA Images with Noise & Distortion
# Simulating distorted alphanumeric CAPTCHAs (e.g., 'A7K9') for training and testing.
# =============================================================================

CHARS = string.ascii_uppercase + string.digits
CHAR_TO_IDX = {c: i for i, c in enumerate(CHARS)}
IDX_TO_CHAR = {i: c for i, c in enumerate(CHARS)}
CAPTCHA_LEN = 4
IMG_WIDTH, IMG_HEIGHT = 120, 40

def generate_captcha(text=None):
    if text is None:
        text = ''.join(random.choices(CHARS, k=CAPTCHA_LEN))
    
    # Create white background image
    img = Image.new('L', (IMG_WIDTH, IMG_HEIGHT), color=255)
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default()
    
    # Draw each character at fixed horizontal slots with slight vertical jitter
    for idx, ch in enumerate(text):
        x = 12 + idx * 26 + random.randint(-2, 2)
        y = 12 + random.randint(-4, 4)
        draw.text((x, y), ch, fill=random.randint(0, 60), font=font)
        
    # Add random distraction lines
    for _ in range(3):
        x1, y1 = random.randint(0, IMG_WIDTH), random.randint(0, IMG_HEIGHT)
        x2, y2 = random.randint(0, IMG_WIDTH), random.randint(0, IMG_HEIGHT)
        draw.line([(x1, y1), (x2, y2)], fill=random.randint(120, 180), width=1)
        
    # Add salt-and-pepper pixel noise
    arr = np.array(img, dtype=np.float32)
    noise = np.random.normal(0, 15, arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    
    return Image.fromarray(arr), text

# Visualize sample generated CAPTCHAs
random.seed(42)
np.random.seed(42)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax in axes:
    sample_img, sample_lbl = generate_captcha()
    ax.imshow(sample_img, cmap='gray')
    ax.set_title(f"Label: {sample_lbl}")
    ax.axis('off')
plt.suptitle("Sample Generated CAPTCHA Images", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 3. Image Preprocessing & Character Segmentation
# Denoising via thresholding and slicing each 4-char CAPTCHA into 4 character crops.
# =============================================================================

def segment_captcha(img):
    # Denoise: binary thresholding to isolate dark foreground text from light noise
    arr = np.array(img, dtype=np.float32)\ / 255.0
    # Invert so text pixels are high values (~1.0) and background is ~0.0
    inv = 1.0 - arr
    inv[inv < 0.35] = 0.0
    
    # Slice horizontally into 4 equal character regions (30x40 each)
    char_crops = []
    slot_w = IMG_WIDTH // CAPTCHA_LEN
    for i in range(CAPTCHA_LEN):
        crop = inv[:, i * slot_w : (i + 1) * slot_w]
        char_crops.append(crop.flatten())
    return char_crops

# Build dataset of 1,500 CAPTCHAs (6,000 individual character samples)
X_chars = []
y_chars = []
captcha_images = []
captcha_labels = []

for _ in range(1500):
    img, label = generate_captcha()
    captcha_images.append(img)
    captcha_labels.append(label)
    crops = segment_captcha(img)
    for c_vec, ch in zip(crops, label):
        X_chars.append(c_vec)
        y_chars.append(CHAR_TO_IDX[ch])

X_chars = np.array(X_chars)
y_chars = np.array(y_chars)

print(f"Total Character Samples: {X_chars.shape[0]} | Feature Vector Size: {X_chars.shape[1]} pixels")

In [ ]:
# =============================================================================
# 4. Train Multi-Layer Perceptron (Neural Network) Character Recognizer
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_chars, y_chars, test_size=0.20, random_state=42, stratify=y_chars
)

# Train MLP Neural Network on flattened 30x40 character patches
mlp_solver = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=30, random_state=42, verbose=False)
mlp_solver.fit(X_train, y_train)

char_preds = mlp_solver.predict(X_test)
print(f"Individual Character Recognition Accuracy: {accuracy_score(y_test, char_preds) * 100:.2f}%")

In [ ]:
# =============================================================================
# 5. End-to-End CAPTCHA Solver Inference & Visualization
# =============================================================================

def solve_captcha(img):
    crops = segment_captcha(img)
    preds = mlp_solver.predict(crops)
    return ''.join(IDX_TO_CHAR[idx] for idx in preds)

# Test full 4-character CAPTCHA solving on 200 newly generated unseen CAPTCHAs
exact_matches = 0
test_samples = []

for i in range(200):
    test_img, true_text = generate_captcha()
    pred_text = solve_captcha(test_img)
    if pred_text == true_text:
        exact_matches += 1
    if i < 6:
        test_samples.append((test_img, true_text, pred_text))

print(f"Full 4-Character CAPTCHA Exact Match Accuracy: {(exact_matches / 200) * 100:.2f}%")

# Display sample predictions
fig, axes = plt.subplots(2, 3, figsize=(12, 5))
for ax, (img, true_txt, pred_txt) in zip(axes.flatten(), test_samples):
    ax.imshow(img, cmap='gray')
    color = 'green' if true_txt == pred_txt else 'red'
    ax.set_title(f"Actual: {true_txt} | Solved: {pred_txt}", color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle("CAPTCHA Solver Predictions on Unseen Challenges", fontsize=14)
plt.tight_layout()
plt.show()